# Visualization of the simulated single-cell datasets

Background: `base_cell_means` is the noise-free mean expression at each cell's path
position, `cell_means` adds Splatter's per-cell gamma noise before Poisson sampling,
and the counts add library-size sampling and dropout on top
(`experiments/synthetic_data/synthetic/data_generation/simulate_splatter.R`).

Prerequisite: the geometry artifacts produced by
`experiments/synthetic_data/scripts/prepare_geometry.py`, i.e.
`{data_root}/{scenario}/geometry/`.


## 0. Setup


In [ ]:
from __future__ import annotations

import sys
import time
from pathlib import Path

from IPython.display import Image, display


def _find_repo_root() -> Path:
    here = Path.cwd().resolve()
    for candidate in (here, here.parent, here.parent.parent):
        if (candidate / "experiments" / "synthetic_data").is_dir():
            return candidate
    raise RuntimeError(
        "Could not locate the repository root (expected experiments/synthetic_data/)."
    )


REPO_ROOT = _find_repo_root()

for _p in (REPO_ROOT, REPO_ROOT / "src"):
    if str(_p) not in sys.path:
        sys.path.insert(0, str(_p))

import matplotlib as mpl
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from matplotlib.collections import LineCollection
from matplotlib.lines import Line2D
from sklearn.manifold import TSNE

from experiments.synthetic_data.common.clean import (
    distance_to_tree,
    load_clean_tree,
    tree_segments,
)
from notebooks.helpers.scenarios import SCENARIO_META
from notebooks.helpers.style import apply_paper_style, savefig

DATA_ROOT = REPO_ROOT / "experiments/synthetic_data/synthetic/data"
OUT_DIR = REPO_ROOT / "notebooks" / "figures" / "sc_data_gallery"
SCENARIOS = ["linear", "bifurcation", "tree"]
SEED = 42
TSNE_MAX_CELLS = 6000
TSNE_PERPLEXITY = 30.0
SHOW_NOISY = True


def show_fig(path: Path) -> None:
    """Display the PNG sibling inline (savefig writes both PNG and SVG)."""
    print("saved:", path)
    display(Image(filename=str(path.with_suffix(".png"))))


apply_paper_style()
OUT_DIR.mkdir(parents=True, exist_ok=True)
print(f"REPO_ROOT = {REPO_ROOT}")
print(f"Saving figures under: {OUT_DIR}")


## 1. Load the geometry artifacts

`X_pca.npy` is the 50-dimensional PCA working space of the preprocessed noisy counts.
`X_clean_pca.npy` / `X_cell_means_pca.npy` map the two clean matrices into the same
space (each with its own filtered row sums). `clean_tree.npz` is the discretised
ground-truth tree built from the `base_cell_means` step means.


In [ ]:
def load_scenario(scenario: str) -> dict:
    """Load the geometry artifacts of one scenario."""
    geo = DATA_ROOT / scenario / "geometry"
    fit = np.load(geo / "pca_fit.npz")
    return {
        "tree": load_clean_tree(geo / "clean_tree.npz"),
        "labels": pd.read_csv(geo / "labels_aligned.csv"),
        "X_noisy": np.load(geo / "X_pca.npy"),
        "X_means": np.load(geo / "X_cell_means_pca.npy"),
        "explained": fit["explained_variance_ratio_"],
    }


DATA = {scenario: load_scenario(scenario) for scenario in SCENARIOS}

PT_VMAX = float(max(DATA[s]["labels"]["PseudotimeGlobal"].max() for s in SCENARIOS))
PALETTE = mpl.colormaps["tab10"]
CMAP = mpl.colormaps["viridis"]
NORM = mpl.colors.Normalize(0.0, PT_VMAX)

for scenario in SCENARIOS:
    entry = DATA[scenario]
    tree = entry["tree"]
    print(
        f"{scenario:12s} {len(entry['labels']):6d} clean cells · "
        f"{len(tree.path_names)} paths · {len(tree.points):4d} tree steps · "
        f"PC1 {100 * entry['explained'][0]:.1f}% / PC2 {100 * entry['explained'][1]:.1f}%"
    )


## 2. Plot helpers


In [ ]:
def path_colors(tree) -> np.ndarray:
    """Categorical color per tree path, keyed by the path number."""
    numbers = [int(name.removeprefix("Path")) for name in tree.path_names]
    return np.array([PALETTE((number - 1) % 10) for number in numbers])


def step_pseudotime(tree, labels: pd.DataFrame, path: int) -> np.ndarray:
    """Mean global pseudotime at each step of one path, ordered by step."""
    rows = labels.loc[labels["Path"] == tree.path_names[path]]
    ptime = rows.groupby("Step")["PseudotimeGlobal"].mean().sort_index().to_numpy()
    assert len(ptime) == tree.n_points[path]
    return ptime


def draw_panel(ax, scenario: str, coords: dict, color_by: str) -> None:
    """Draw one embedding panel with the tree, clean cells and noisy background."""
    entry = DATA[scenario]
    tree, labels = entry["tree"], entry["labels"]
    colors = path_colors(tree)
    mapping = {name: i for i, name in enumerate(tree.path_names)}
    cells_path = labels["Path"].map(mapping).to_numpy()
    cells_ptime = labels["PseudotimeGlobal"].to_numpy()

    if SHOW_NOISY:
        ax.scatter(
            coords["noisy"][:, 0],
            coords["noisy"][:, 1],
            s=1.0,
            c="0.75",
            alpha=0.3,
            linewidths=0,
            rasterized=True,
            zorder=1,
        )

    idx = coords["means_idx"]
    if color_by == "path":
        scatter_kw = {"c": colors[cells_path[idx]]}
    else:
        scatter_kw = {"c": cells_ptime[idx], "cmap": CMAP, "norm": NORM}
    ax.scatter(
        coords["means"][:, 0],
        coords["means"][:, 1],
        s=3.0,
        alpha=0.5,
        linewidths=0,
        rasterized=True,
        zorder=2,
        **scatter_kw,
    )

    for path in range(len(tree.path_names)):
        start = int(tree.point_start[path])
        stop = start + int(tree.n_points[path])
        pts = coords["tree"][start:stop]
        ptime = step_pseudotime(tree, labels, path)
        segments = np.stack([pts[:-1], pts[1:]], axis=1)
        if color_by == "path":
            segment_colors = np.tile(colors[path], (len(segments), 1))
            face, edge = "white", colors[path]
        else:
            segment_colors = CMAP(NORM(0.5 * (ptime[:-1] + ptime[1:])))
            face, edge = CMAP(NORM(ptime[::20])), "white"
        ax.add_collection(
            LineCollection(
                segments,
                colors=segment_colors,
                linewidths=2.0,
                zorder=4,
                capstyle="round",
            )
        )
        ax.scatter(
            pts[::20, 0],
            pts[::20, 1],
            s=8,
            facecolor=face,
            edgecolor=edge,
            linewidths=0.6,
            zorder=5,
        )


def style_panel(
    ax, scenario: str, kind: str, *, show_xlabel: bool, show_ylabel: bool
) -> None:
    ax.set_xticks([])
    ax.set_yticks([])
    for spine in ax.spines.values():
        spine.set_linewidth(0.6)
    if kind == "pca":
        evr = DATA[scenario]["explained"]
        if show_xlabel:
            ax.set_xlabel(f"PC1 ({100 * evr[0]:.1f}%)")
        if show_ylabel:
            ax.set_ylabel(f"PC2 ({100 * evr[1]:.1f}%)")
    else:
        if show_xlabel:
            ax.set_xlabel("t-SNE 1")
        if show_ylabel:
            ax.set_ylabel("t-SNE 2")


In [ ]:
def make_gallery(kind: str, coords: dict, out_path: Path) -> Path:
    """Save the gallery figure for one embedding and return the SVG path."""
    fig, axs = plt.subplots(
        2, len(SCENARIOS), figsize=(4.6 * len(SCENARIOS), 9.4), squeeze=False
    )
    for col, scenario in enumerate(SCENARIOS):
        for row, color_by in enumerate(("path", "pseudotime")):
            ax = axs[row, col]
            draw_panel(ax, scenario, coords[scenario], color_by)
            style_panel(ax, scenario, kind, show_xlabel=row == 1, show_ylabel=col == 0)
            if row == 0:
                meta = SCENARIO_META[scenario]
                ax.set_title(f"{meta['label']}\n{meta['description']}", fontsize=10)

    tree_all = DATA["tree"]["tree"]
    colors_all = path_colors(tree_all)
    handles = [
        Line2D(
            [],
            [],
            color=color,
            lw=1.4,
            marker="o",
            markersize=3.5,
            markerfacecolor="white",
            markeredgecolor=color,
            markeredgewidth=0.6,
        )
        for color in colors_all
    ]
    handles += [
        Line2D([], [], color="0.75", marker=".", lw=0, markersize=6, alpha=0.6),
        Line2D([], [], color="0.3", marker="o", lw=0, markersize=3.5),
        Line2D(
            [],
            [],
            color="0.3",
            marker="o",
            lw=0,
            markersize=3.5,
            markerfacecolor="white",
        ),
    ]
    labels = list(tree_all.path_names) + [
        "noisy counts",
        "clean cells (cell_means)",
        "tree steps (base_cell_means)",
    ]
    fig.legend(
        handles,
        labels,
        loc="lower center",
        ncol=5,
        bbox_to_anchor=(0.5, -0.03),
        frameon=False,
    )

    sm = mpl.cm.ScalarMappable(norm=NORM, cmap=CMAP)
    sm.set_array([])
    fig.colorbar(
        sm,
        ax=axs[1, :],
        label="global pseudotime",
        fraction=0.025,
        pad=0.01,
        location="right",
    )
    method = (
        "PCA (working space)"
        if kind == "pca"
        else f"t-SNE (perplexity {TSNE_PERPLEXITY:g})"
    )
    fig.suptitle(
        f"Splatter clean targets — {method}\n"
        "top: colored by path · bottom: colored by pseudotime",
        fontsize=12,
    )
    return savefig(fig, out_path)


## 3. Embedding coordinates

The PCA panels are a direct slice of the stored working space, so they need no fitting;
their axis labels show the explained variance of the Stage I PCA fit. The t-SNE panels
are independent per-scenario fits on the union of the tree steps and capped subsamples
of the clean and noisy cells, so distances and orientations are not comparable across
panels. Set `TSNE_MAX_CELLS = 0` in the setup cell to embed all cells (the three panels
take about a minute at the default cap).


In [ ]:
def _subsample(n: int, cap: int, rng: np.random.Generator) -> np.ndarray:
    if cap <= 0 or cap >= n:
        return np.arange(n)
    return np.sort(rng.choice(n, size=cap, replace=False))


def embedding_coords(scenario: str, kind: str) -> dict:
    """2D coordinates of the tree steps, the clean cells and the noisy cells."""
    entry = DATA[scenario]
    tree = entry["tree"]
    if kind == "pca":
        return {
            "tree": tree.points.astype(np.float64)[:, :2],
            "means": entry["X_means"][:, :2],
            "means_idx": np.arange(len(entry["X_means"])),
            "noisy": entry["X_noisy"][:, :2],
            "noisy_idx": np.arange(len(entry["X_noisy"])),
        }
    rng = np.random.default_rng(SEED)
    idx_means = _subsample(len(entry["X_means"]), TSNE_MAX_CELLS, rng)
    idx_noisy = _subsample(len(entry["X_noisy"]), TSNE_MAX_CELLS, rng)
    X = np.vstack(
        [tree.points, entry["X_means"][idx_means], entry["X_noisy"][idx_noisy]]
    )
    Y = TSNE(
        n_components=2,
        perplexity=TSNE_PERPLEXITY,
        init="pca",
        random_state=SEED,
        n_jobs=-1,
    ).fit_transform(X)
    n_tree = len(tree.points)
    n_means = len(idx_means)
    return {
        "tree": Y[:n_tree],
        "means": Y[n_tree : n_tree + n_means],
        "means_idx": idx_means,
        "noisy": Y[n_tree + n_means :],
        "noisy_idx": idx_noisy,
    }


COORDS: dict[str, dict[str, dict]] = {"pca": {}, "tsne": {}}
for scenario in SCENARIOS:
    COORDS["pca"][scenario] = embedding_coords(scenario, "pca")
print("PCA panels use the stored working space; no fitting required.")


In [ ]:
for scenario in SCENARIOS:
    tick = time.time()
    COORDS["tsne"][scenario] = embedding_coords(scenario, "tsne")
    print(f"{scenario:12s} t-SNE fitted in {time.time() - tick:.0f}s")


## 4. PCA gallery

PC1 / PC2 of the working space. The clean cells trace the tree with the expected lateral
spread, and the noisy counts form a wider cloud around it.


In [ ]:
show_fig(make_gallery("pca", COORDS["pca"], OUT_DIR / "01_pca_gallery"))


## 5. t-SNE gallery

t-SNE resolves the branches more clearly than the linear projection


In [ ]:
show_fig(make_gallery("tsne", COORDS["tsne"], OUT_DIR / "02_tsne_gallery"))
